In [1]:
# =====================================================================
#  WhisperLive + Flask + ngrok — real-time HINDI / HINGLISH transcription
#  One cell. Kaggle settings: Internet = ON, Accelerator = GPU T4 (required for large-v3)
# =====================================================================

NGROK_AUTH_TOKEN = "2Rq48TXamsdXdrfkiTJHuTIkVMq_3DHJmnQppph2KuNUtkaxh"
MODEL_SIZE       = "large-v3"    # large-v3 / medium / small  (Hindi needs medium+)
LANGUAGE         = "hi"          # "hi" = Hindi. Do NOT use None for mixed Hindi-English.
SCRIPT           = "hinglish"    # "hinglish" -> Roman  |  "hindi" -> देवनागरी
USE_VAD          = True
FLASK_PORT       = 5000
WL_PORT          = 9090

import os, re, sys, json, time, uuid, socket, threading, subprocess

def sh(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True)

# ---------------------------------------------------------------- 1. deps
sh("apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y portaudio19-dev ffmpeg > /dev/null 2>&1")
sh(f"{sys.executable} -m pip install -q whisper-live flask flask-sock simple-websocket "
   f"websocket-client pyngrok indic-transliteration")

# pre-download ctranslate2 weights (~3 GB for large-v3) so the first click isn't a stall
try:
    from huggingface_hub import snapshot_download
    snapshot_download(f"Systran/faster-whisper-{MODEL_SIZE}")
    print("model cached:", MODEL_SIZE)
except Exception as e:
    print("model pre-download skipped:", e)

# ------------------------------------------------- 2. WhisperLive server
LAUNCHER = f'''
import inspect
from whisper_live.server import TranscriptionServer
srv = TranscriptionServer()
opts = dict(backend="faster_whisper", max_clients=4, max_connection_time=86400,
            single_model=False, cache_path="/root/.cache/whisper-live/")
p = inspect.signature(srv.run).parameters
kw = {{k: v for k, v in opts.items() if k in p}}
print("run() kwargs:", kw, flush=True)
srv.run("0.0.0.0", port={WL_PORT}, **kw)
'''
open("wl_server.py", "w").write(LAUNCHER)

WL_LOG = "wl_server.log"
wl_proc = subprocess.Popen([sys.executable, "wl_server.py"],
                           stdout=open(WL_LOG, "wb"), stderr=subprocess.STDOUT)

def wait_port(port, timeout=300):
    t0 = time.time()
    while time.time() - t0 < timeout:
        with socket.socket() as s:
            s.settimeout(1)
            if s.connect_ex(("127.0.0.1", port)) == 0:
                return True
        if wl_proc.poll() is not None:
            print(open(WL_LOG).read()[-3000:]); raise RuntimeError("WhisperLive server died")
        time.sleep(1)
    return False

print("waiting for WhisperLive on :%d ..." % WL_PORT)
print("WhisperLive up:", wait_port(WL_PORT))

def tail_logs(n=40):
    print("".join(open(WL_LOG, errors="ignore").readlines()[-n:]))

# ------------------------------------------- 3. Devanagari -> Hinglish
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

def to_hinglish(t):
    if not t or not re.search(r"[\u0900-\u097F]", t):
        return t                                     # already Roman / empty
    out = transliterate(t, sanscript.DEVANAGARI, sanscript.ITRANS).lower()
    out = (out.replace("~n", "n").replace("~m", "n").replace(".n", "n")
              .replace(".h", "").replace("|", ".").replace("r^i", "ri"))
    out = re.sub(r"([aiu])\1+", r"\1", out)           # kyaa -> kya, hii -> hi
    return re.sub(r"\s+", " ", out).strip()

def localize(raw):
    """Rewrite segment text in the chosen script before forwarding to the browser."""
    if SCRIPT != "hinglish":
        return raw
    try:
        m = json.loads(raw)
    except Exception:
        return raw
    if isinstance(m.get("segments"), list):
        for s in m["segments"]:
            if isinstance(s.get("text"), str):
                s["text"] = to_hinglish(s["text"])
        return json.dumps(m)
    return raw

# ------------------------------------------------------- 4. Flask + relay
from flask import Flask, Response
from flask_sock import Sock
import websocket as wsclient          # websocket-client

app = Flask(__name__)
app.config["SOCK_SERVER_OPTIONS"] = {"ping_interval": 25}
sock = Sock(app)

def wl_config(uid):
    return {
        "uid": uid, "language": LANGUAGE, "task": "transcribe", "model": MODEL_SIZE,
        "use_vad": USE_VAD, "max_clients": 4, "max_connection_time": 86400,
        "send_last_n_segments": 10, "no_speech_thresh": 0.45, "clip_audio": False,
        "same_output_threshold": 10, "enable_translation": False, "target_language": "hi",
        "initial_prompt": "यह हिंदी बातचीत है।",
    }

@sock.route("/ws")
def relay(browser):
    uid = str(uuid.uuid4())
    try:
        wl = wsclient.create_connection(f"ws://127.0.0.1:{WL_PORT}", timeout=15)
    except Exception as e:
        browser.send(json.dumps({"message": "ERROR", "detail": str(e)})); return
    wl.send(json.dumps(wl_config(uid)))
    stop = threading.Event()

    def pump():                       # WhisperLive -> browser
        while not stop.is_set():
            try:
                msg = wl.recv()
            except Exception:
                break
            if not msg:
                break
            try:
                txt = msg if isinstance(msg, str) else msg.decode("utf-8", "ignore")
                browser.send(localize(txt))
            except Exception:
                break
        stop.set()

    threading.Thread(target=pump, daemon=True).start()
    try:
        while not stop.is_set():                   # browser -> WhisperLive
            data = browser.receive(timeout=1)
            if data is None:
                continue
            if isinstance(data, (bytes, bytearray)):
                wl.send_binary(bytes(data))        # raw float32 PCM @16 kHz mono
            else:
                if json.loads(data or "{}").get("type") == "stop":
                    wl.send_binary(b"END_OF_AUDIO"); break
    except Exception:
        pass
    finally:
        stop.set()
        try: wl.close()
        except Exception: pass

PAGE = r"""<!doctype html><html><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1"><title>WhisperLive Hindi</title>
<style>
 body{font-family:system-ui,'Noto Sans Devanagari',sans-serif;max-width:820px;margin:32px auto;
      padding:0 16px;background:#0f1115;color:#e6e6e6}
 h1{font-size:20px;margin-bottom:4px}.meta{color:#8b93a7;font-size:13px;margin-bottom:20px}
 button{font-size:16px;padding:12px 22px;border:0;border-radius:10px;cursor:pointer;background:#3b82f6;color:#fff}
 button.rec{background:#ef4444}button:disabled{opacity:.5;cursor:default}
 #status{margin-left:12px;font-size:14px;color:#8b93a7}
 #out{margin-top:22px;padding:16px;min-height:220px;max-height:52vh;overflow:auto;
      background:#171a21;border:1px solid #262b36;border-radius:12px;font-size:18px;line-height:1.7}
 .done{color:#e6e6e6}.live{color:#f59e0b}
 .dot{display:inline-block;width:9px;height:9px;border-radius:50%;background:#ef4444;margin-right:6px;animation:p 1s infinite}
 @keyframes p{50%{opacity:.25}}
</style></head><body>
<h1>WhisperLive — Hindi real-time transcription</h1>
<div class="meta">model <b>__MODEL__</b> · language <b>__LANG__</b> · script <b>__SCRIPT__</b> · VAD <b>__VAD__</b></div>
<button id="btn">🎙️ Start listening</button><span id="status">idle</span>
<div id="out"><span style="color:#5b6478">बोलिए — transcript yahan aayega…</span></div>
<script>
const btn=document.getElementById('btn'),st=document.getElementById('status'),out=document.getElementById('out');
let ws,ctx,stream,node,src,running=false,segs=new Map();
const esc=s=>s.replace(/[&<>]/g,c=>({'&':'&amp;','<':'&lt;','>':'&gt;'}[c]));
function render(){
  const a=[...segs.values()].sort((x,y)=>parseFloat(x.start)-parseFloat(y.start));
  out.innerHTML = a.length? a.map(s=>`<span class="${s.completed?'done':'live'}">${esc(s.text)}</span>`).join(' ')
                          : '<span style="color:#5b6478">listening…</span>';
  out.scrollTop=out.scrollHeight;
}
function down(buf,from,to){
  if(from===to) return buf;
  const r=from/to,len=Math.round(buf.length/r),o=new Float32Array(len);let p=0;
  for(let i=0;i<len;i++){const n=Math.round((i+1)*r);let s=0,c=0;
    for(let j=p;j<n&&j<buf.length;j++){s+=buf[j];c++;}o[i]=c?s/c:0;p=n;}
  return o;
}
async function start(){
  segs.clear();render();st.textContent='connecting…';
  ws=new WebSocket((location.protocol==='https:'?'wss':'ws')+'://'+location.host+'/ws');
  ws.binaryType='arraybuffer';
  ws.onmessage=e=>{let m;try{m=JSON.parse(e.data)}catch(_){return}
    if(m.status==='WAIT'){st.textContent='server busy, wait ~'+m.message+' min';return}
    if(m.message==='SERVER_READY'){st.innerHTML='<span class="dot"></span>listening ('+(m.backend||'')+')';return}
    if(m.message==='ERROR'){st.textContent='error: '+(m.detail||'');return}
    if(m.message==='DISCONNECT'){st.textContent='server closed session';stop();return}
    if(m.language){st.innerHTML='<span class="dot"></span>listening · lang '+m.language;}
    if(m.segments){m.segments.forEach(s=>segs.set(String(parseFloat(s.start).toFixed(2)),s));render();}
  };
  ws.onclose=()=>{if(running)stop()};
  ws.onerror=()=>{st.textContent='websocket error'};
  stream=await navigator.mediaDevices.getUserMedia({audio:{channelCount:1,sampleRate:16000,
          echoCancellation:true,noiseSuppression:true,autoGainControl:true}});
  ctx=new (window.AudioContext||window.webkitAudioContext)({sampleRate:16000});
  await ctx.resume();
  src=ctx.createMediaStreamSource(stream);
  node=ctx.createScriptProcessor(4096,1,1);
  node.onaudioprocess=e=>{
    if(!ws||ws.readyState!==1)return;
    let d=e.inputBuffer.getChannelData(0);
    if(ctx.sampleRate!==16000)d=down(d,ctx.sampleRate,16000);
    ws.send(new Float32Array(d).buffer);
  };
  src.connect(node);node.connect(ctx.destination);
  running=true;btn.textContent='⏹ Stop';btn.classList.add('rec');
}
function stop(){
  running=false;btn.textContent='🎙️ Start listening';btn.classList.remove('rec');st.textContent='stopped';
  try{ws.send(JSON.stringify({type:'stop'}))}catch(_){}
  setTimeout(()=>{try{ws.close()}catch(_){}} ,400);
  try{node.disconnect();src.disconnect();ctx.close()}catch(_){}
  try{stream.getTracks().forEach(t=>t.stop())}catch(_){}
}
btn.onclick=async()=>{btn.disabled=true;try{running?stop():await start()}catch(e){st.textContent=e.message}
  btn.disabled=false;};
</script></body></html>"""

@app.route("/")
def index():
    html = (PAGE.replace("__MODEL__", MODEL_SIZE)
                .replace("__LANG__", LANGUAGE or "auto")
                .replace("__SCRIPT__", SCRIPT)
                .replace("__VAD__", str(USE_VAD)))
    return Response(html, mimetype="text/html")

threading.Thread(target=lambda: app.run(host="0.0.0.0", port=FLASK_PORT,
                 threaded=True, debug=False, use_reloader=False), daemon=True).start()
time.sleep(2)

# ------------------------------------------------------------- 5. ngrok
from pyngrok import ngrok
if not NGROK_AUTH_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        NGROK_AUTH_TOKEN = UserSecretsClient().get_secret("NGROK_AUTH_TOKEN")
    except Exception:
        pass
assert NGROK_AUTH_TOKEN, "Set NGROK_AUTH_TOKEN (ngrok.com -> Your Authtoken)"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
try: ngrok.kill()
except Exception: pass
public_url = ngrok.connect(FLASK_PORT, "http").public_url.replace("http://", "https://")

print("\n" + "=" * 70)
print("  OPEN THIS URL, click 'Start listening', allow the mic:")
print("  ", public_url)
print("=" * 70)
print("  SCRIPT = 'hindi' for देवनागरी, 'hinglish' for Roman — re-run cell to switch")
print("  tail_logs()  -> WhisperLive server log")
print("  wl_proc.kill(); ngrok.kill()  -> shut everything down")

$ apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y portaudio19-dev ffmpeg > /dev/null 2>&1
$ /usr/bin/python3 -m pip install -q whisper-live flask flask-sock simple-websocket websocket-client pyngrok indic-transliteration
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 16.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which i

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

model.bin:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

model cached: large-v3
waiting for WhisperLive on :9090 ...
WhisperLive up: True
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.19.2.2:5000
Press CTRL+C to quit


                                                                                                    
  OPEN THIS URL, click 'Start listening', allow the mic:
   https://351c-35-238-211-48.ngrok-free.app
  SCRIPT = 'hindi' for देवनागरी, 'hinglish' for Roman — re-run cell to switch
  tail_logs()  -> WhisperLive server log
  wl_proc.kill(); ngrok.kill()  -> shut everything down


127.0.0.1 - - [10/Aug/2026 16:23:58] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/Aug/2026 16:23:59] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [10/Aug/2026 16:26:17] "GET /ws HTTP/1.1" 200 -
